In [1]:
%load_ext autoreload
%autoreload 2

import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from moneyball.config import COMPETITION_ID, SEASON_ID
from moneyball.data_loader import StatsBombClient
from moneyball.processing import events_to_dataframe,lineups_to_appearances

client = StatsBombClient()
SAMPLE_MATCH = 3879608  # Hellas Verona - Bologna, explored in stage 1


def raises(error_type, func, *args):
    """Return True if ``func(*args)`` raises ``error_type``."""
    try:
        func(*args)
    except error_type:
        return True
    return False

In [2]:
from moneyball.processing import matches_to_dataframe

# Hand-made input: nested teams, an extra key to drop, dates out of order.
fake_matches = [
    {
        "match_id": 2,
        "match_date": "2016-01-10",
        "match_week": 2,
        "home_team": {"home_team_name": "Napoli", "home_team_id": 1},
        "away_team": {"away_team_name": "Juventus", "away_team_id": 2},
        "home_score": 1,
        "away_score": 2,
        "referee": {"name": "Not needed"},
    },
    {
        "match_id": 1,
        "match_date": "2016-01-03",
        "match_week": 1,
        "home_team": {"home_team_name": "Juventus", "home_team_id": 2},
        "away_team": {"away_team_name": "Napoli", "away_team_id": 1},
        "home_score": 0,
        "away_score": 0,
        "referee": {"name": "Not needed"},
    },
]
fake = matches_to_dataframe(fake_matches)
fake

,match_id,match_date,match_week,home_team,away_team,home_score,away_score
0,1,2016-01-03,1,Juventus,Napoli,0,0
1,2,2016-01-10,2,Napoli,Juventus,1,2


In [3]:
df = events_to_dataframe(client.get_events(3879608), 3879608)
df.shape                      
df["under_pressure"].unique()

array([False,  True])

In [28]:
spells = lineups_to_appearances(client.get_lineups(3879608), 3879608, 95.133)
spells[["player_name", "position", "minutes"]].head(15)


,player_name,position,minutes
0,Emil Hallfreðsson,Left Center Midfield,95.133000
1,Giampaolo Pazzini,Center Forward,95.133000
2,Jacopo Sala,Right Center Midfield,91.233000
3,Luca Rossettini,Right Back,95.133000
4,Adam Masina,Left Back,95.133000
5,Antonio Mirante,Goalkeeper,95.133000
6,Samuel Souprayen,Left Back,78.450000
7,Emanuele Giaccherini,Left Wing,70.750000
8,Artur Ioniţă,Right Center Midfield,28.666333
9,Filip Helander,Left Center Back,95.133000


In [5]:
from moneyball.processing import clock_to_minutes, match_end_minute

assert clock_to_minutes("00:00") == 0
assert clock_to_minutes("90:30") == 90.5
assert abs(clock_to_minutes("59:44") - 59.733) < 1e-3

half_ends = pd.DataFrame(
    {
        "type_name": ["Half End", "Half End", "Pass", "Half End", "Half End"],
        "period": [1, 1, 2, 2, 2],
        "minute": [46, 46, 94, 95, 95],
        "second": [12, 12, 50, 8, 8],
    }
)
assert abs(match_end_minute(half_ends) - (95 + 8 / 60)) < 1e-9, "must use the last period"
assert raises(ValueError, match_end_minute, half_ends[half_ends["type_name"] == "Pass"]), (
    "no Half End must raise ValueError"
)
print("T2.3 clock and end minute OK")

T2.3 clock and end minute OK


In [6]:
from moneyball.processing import lineups_to_appearances

FAKE_MATCH = 999

def spell(position, start, end):
    """One entry of a lineup ``positions`` list."""
    return {"position": position, "from": start, "to": end}


fake_lineups = [
    {
        "team_name": "Napoli",
        "lineup": [
            # Full match at one position.
            {
                "player_id": 1,
                "player_name": "Striker",
                "positions": [spell("Center Forward", "00:00", None)],
            },
            # Two positions: 60' on the wing, then 30' up front -> main position = wing.
            {
                "player_id": 2,
                "player_name": "Winger",
                "positions": [
                    spell("Left Wing", "00:00", "60:00"),
                    spell("Center Forward", "60:00", None),
                ],
            },
            # Unused substitute: no spell -> must not appear.
            {"player_id": 3, "player_name": "Bench", "positions": []},
            # Came on at 75:30.
            {
                "player_id": 4,
                "player_name": "Sub",
                "positions": [spell("Right Back", "75:30", None)],
            },
        ],
    },
    {
        "team_name": "Juventus",
        "lineup": [
            # Sent off at 30:00.
            {
                "player_id": 5,
                "player_name": "Red card",
                "positions": [spell("Goalkeeper", "00:00", "30:00")],
            },
        ],
    },
]
FAKE_END = 90.0
fake_app = lineups_to_appearances(fake_lineups, FAKE_MATCH, FAKE_END).set_index("player_id")
fake_app

,match_id,player_name,team_name,position,minutes
player_id,,,,,
1,999,Striker,Napoli,Center Forward,90.0
2,999,Winger,Napoli,Left Wing,90.0
4,999,Sub,Napoli,Right Back,14.5
5,999,Red card,Juventus,Goalkeeper,30.0
